# Adaptive Sampling

We wish to adaptively sample a random process to better "learn" the underlying function.

Simply, until we are satisfied with the total variance of the posterior, we (iteratively) update our posterior with the point in our domain associated with the highest variance.


In [ ]:
import matplotlib.axes as axs
import matplotlib.pyplot as plt
import numpy as np
import typing

import sys
sys.path.insert(0, '../')

import UncertainSCI.gp as gp


ALPHA = 0.6

N_PTS = 200

XLIM = (-5, 5)
YLIM = (-3, 3)
BINS = 200


Define our domain and some plotting helpers:

In [ ]:
X, dx = np.linspace(*XLIM, N_PTS, retstep=True)

def draw_distribution(ax: axs.Axes, g: gp.ScalarGaussianProcess, which='posterior'):
    if not (which == 'posterior' or which == 'prior'):
        raise ValueError("`which` parameter must be 'posterior' (default) or 'prior'")

    x = np.linspace(*XLIM, N_PTS)
    y = np.linspace(*YLIM, N_PTS)
    m = g.mu_posterior(x) if which == 'posterior' else g.mu(x)
    v = np.diag(g.k_posterior(x)) if which == 'posterior' else np.diag(g.k(x))

    vmap = (1 / (2 * np.pi * v[:, None]) * np.exp(-(m[:, None] - y[None, :])**2 / (2 * v[:, None]))).T

    ax.pcolormesh(
        x, y, vmap / np.max(vmap, axis=0),
        cmap='Greys',
        alpha=ALPHA
    )

def draw_plot(ax: axs.Axes, f: typing.Callable, g: gp.ScalarGaussianProcess):
    x = np.linspace(*XLIM, 1000)
    ax.plot(x, f(x), color='tab:red', linewidth=4, zorder=10.)

    draw_distribution(ax, g)
    ax.plot(X, g.mu_posterior(X), color='black')
    ax.plot(X, g.mu_posterior(X) + np.sqrt(np.diag(g.k_posterior(X))), color='black')
    ax.plot(X, g.mu_posterior(X) - np.sqrt(np.diag(g.k_posterior(X))), color='black')
    ax.plot(X, g.sample_posterior(X, 5), alpha=ALPHA)

    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)


## True Function and Noisy Observations

Define the true function and a function that yields noisy observations of that true function.

In particular, for this example, let the true function $f: \mathbb{R} \rightarrow \mathbb{R}$.  Then noisy observations $\hat{f}(x) = f(x) + \epsilon$ of the true function.  As defined below, $\epsilon \sim \text{N}(0, \sigma^2)$ where $\sigma^2 \sim \text{LogNormal}(-1, 0.8)$.  In general, however, GP methods are able to handle noise that is itself a function of the spatial coordinate, or even more exotic scenarios.

In [ ]:
def f(x: np.ndarray | float):
    return 1/3 * (x + 2) * (x - 2) * (x) * np.exp(-x**2 / 5)

def f_random(x: np.ndarray | float):
    fx = f(x)
    s = sigma(fx)
    return fx + (s.flatten() * np.random.normal(0, 1, s.size)).reshape(s.shape), s

def sigma(fx: np.ndarray):
    # (mu, sigma) = (-1, 0.8) chosen simply for looking nice
    return np.random.lognormal(-1, 0.8, fx.shape)


We plot the true function below:

In [ ]:
plt.plot(X, f(X), linewidth=4, color='tab:red', label='Truth')
plt.xlim(*XLIM)
plt.ylim(*YLIM)
plt.show()


## Defining GP and Prior Mean and Covariance
Define the prior Gaussian process and create the ScalarGaussianProcess object:

In [ ]:
mu = gp.wrapper.ScalarFunction(dim=1, f=lambda x: np.zeros_like(x))  # zero mean
k = gp.kernel.SquareExponential(dim=1, gamma=1.)  # square exponential covariance kernel with scale 1
g = gp.ScalarGaussianProcess(mu, k)  # GP defined from these mu, k


We can plot some realizations of the prior:

In [ ]:
plt.plot(X, g.sample_prior(X, 5))
plt.xlim(*XLIM)
plt.ylim(*YLIM)
plt.show()


We can plot the statistics of of the prior.

Note that, below, the gray shading for any $x$ corresponds to the marginal density of the GP at the corresponding $x$ coordinate.  The black lines top-to-bottom correspond to the 1-$\sigma$, mean, and -1-$\sigma$ levels.

In [ ]:
fig, ax = plt.subplots(1, 1)

draw_distribution(ax, g, which='prior')
ax.plot(X, g.mu(X), color='black')
ax.plot(X, g.mu(X) + np.sqrt(np.diag(g.k(X))), color='black')
ax.plot(X, g.mu(X) - np.sqrt(np.diag(g.k(X))), color='black')

plt.xlim(*XLIM)
plt.ylim(*YLIM)
plt.show()


## Conditioned GP and Posterior Mean and Covariance
Now we condition the GP on realizations of the random function $\hat{f}$ (i.e., `f_random` above):

In [ ]:
N_STRT = 3
x_obs = XLIM[0] + np.ptp(XLIM) * np.random.rand(N_STRT)
y_obs, s_obs = f_random(x_obs)
g.condition(x_obs, y_obs, s_obs)


Before conducting any iterative procedure, we ought to plot our initial "guess" at the true function given the observation pairs $(x_{obs}, y_{obs})$ associated with variances $\sigma_{obs}$, which we do below:

In [ ]:
fig, ax = plt.subplots(1, 1)
draw_plot(ax, f, g)
ax.errorbar(x_obs, y_obs, yerr=s_obs, color='black', linestyle='none', marker='.', capsize=4)
plt.show()


We now iteratively update our posterior with observations at the coordinate associated with the highest variance.

In the figures below, the green vertical bar indicates the *coordinate* at which to choose the next sample.  In the figure that follows, the sample chosen at this location is shown with its associated (sampled) variance.

In [ ]:
N_RUN = 10
PLOT_EVERY = 1

for i in range(N_RUN):
    x = X[np.argsort(np.diag(g.k_posterior(X)))[-1:]]
    y, s = f_random(x)

    if (i + 1) % PLOT_EVERY == 0:  # plot showing where to sample
        print(f'Sample {i + 1}:')
        
        fig, ax = plt.subplots(1, 1)
        draw_plot(ax, f, g)
        ax.errorbar(x_obs, y_obs, yerr=s_obs, color='black', linestyle='none', marker='.', capsize=4)
        ax.vlines(x, *YLIM, color="#00ff00", zorder=10.)
        plt.show()

    x_obs = np.concatenate((x_obs, x))
    y_obs = np.concatenate((y_obs, y))    
    s_obs = np.concatenate((s_obs, s))
    g.condition(x_obs, y_obs, s_obs)

    if (i + 1) % PLOT_EVERY == 0:  # plot showing updated posterior
        fig, ax = plt.subplots(1, 1)
        draw_plot(ax, f, g)
        ax.errorbar(x_obs[:-1], y_obs[:-1], yerr=s_obs[:-1], color='black', linestyle='none', marker='.', capsize=4)
        ax.errorbar(x_obs[-1], y_obs[-1], yerr=s_obs[-1], color='#00ff00', linestyle='none', marker='.', capsize=4, zorder=10.)
        plt.show()

        print('\n' * 3, end='')


Finally, we can plot the true function (red) versus the approximation of that function constructed via the GP and adaptive sampling process (black):

In [ ]:
fig, ax = plt.subplots(1, 1)
x = np.linspace(*XLIM, 1000)
ax.plot(x, f(x), linewidth=4, color='tab:red', label='Truth')
ax.plot(x, g.mu_posterior(x), color='black', label='Approximant')
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
plt.legend()
plt.show()
